# 📝 에이전트 품질 과제 LV2(응용)

> LV1 에서 익힌 조각들을 **조합**합니다: 성찰 루프 조립, 분석+생성 결합, 콜백 부착, 프롬프트 버전 교체, 재시도·폴백.

## 풀이 방법
1. 맨 위 **준비 셀들**을 먼저 실행하세요(성찰 부품·분석 도구·프롬프트 레지스트리가 제공됩니다).
2. 각 문제의 **답안 셀**을 채우고 **자가채점 셀**로 확인하세요.

- 데이터: `data/gym_members.csv`(헬스장 회원). 제공된 `generate`·`critique`·`revise`·`summarize_by`·`get_prompt` 를 활용하세요.

화이팅!

아래 준비 셀들을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우
load_dotenv("../../.env") # 교안 폴더 안의 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 오늘 쓸 모델 - 실행만 하세요(LangChain 기본 단원에서 만든 것과 같습니다).
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

print('모델 준비 완료:', type(model).__name__)

In [ ]:
# [제공 코드] 성찰 루프 부품 - 지난 강의에서 만든 생성·비평·수정 함수입니다(실행만 하세요).
from pydantic import BaseModel, Field


class Critique(BaseModel):
    score: int = Field(ge=1, le=10, description='1~10 종합 점수')
    issues: list[str] = Field(description='개선점 목록(짧게)')


_GEN_SYSTEM = "너는 데이터 분석 리포트 작성자다. 주어진 수치 요약을 바탕으로 핵심 인사이트를 한국어 세 문장으로 써라."
_CRITIC_SYSTEM = ("너는 깐깐한 리포트 편집자다. 아래 리포트를 평가하라. 구체적 수치 인용·해석의 명확성·실행 제안 유무를 "
                  "기준으로 1~10점을 매기고, 개선점을 항목으로 지적하라.")
_REVISE_SYSTEM = "너는 리포트 작성자다. 아래 [리포트]를 [개선점]을 모두 반영해 다시 써라. 한국어 세 문장을 유지하라."


def generate(summary):
    """수치 요약으로 리포트 초안을 쓴다."""
    r = model.invoke([{'role': 'system', 'content': _GEN_SYSTEM},
                      {'role': 'user', 'content': summary}])
    return r.text


def critique(report):
    """리포트를 구조화된 출력(Critique: 점수·개선점)으로 평가한다."""
    critic_model = model.with_structured_output(Critique)
    return critic_model.invoke([{'role': 'system', 'content': _CRITIC_SYSTEM},
                                {'role': 'user', 'content': report}])


def revise(report, issues):
    """리포트와 개선점을 받아 다시 쓴다."""
    user = f'[리포트]\n{report}\n\n[개선점]\n' + '\n'.join(f'- {x}' for x in issues)
    r = model.invoke([{'role': 'system', 'content': _REVISE_SYSTEM},
                      {'role': 'user', 'content': user}])
    return r.text


In [ ]:
# [제공 코드] 집계 요약을 돌려주는 분석 도구입니다 - 실행만 하세요.
import pandas as pd


def summarize_by(df, group_col, value_col):
    """group_col 별 value_col 평균을 내림차순 문자열로 요약한다(리포트 입력용)."""
    s = df.groupby(group_col)[value_col].mean().sort_values(ascending=False).round(1)
    parts = [f"{k} {v}" for k, v in s.items()]
    return f"{group_col}별 평균 {value_col}: " + ", ".join(parts)


In [ ]:
# [제공 코드] 프롬프트 버전 레지스트리(로컬) - 실행만 하세요.
# 프롬프트를 코드가 아니라 파일(data/prompts.yaml)에서 불러오면, 코드 배포 없이 프롬프트만 교체할 수 있습니다.
# Langfuse 를 쓰면 langfuse.get_prompt(이름, version=번호) 가 똑같은 일을 서버에서 해 줍니다.
from pathlib import Path

import yaml

# 노트북 위치에 따라 data 폴더가 몇 단계 위인지 달라집니다(일차 폴더 / 교안 폴더 / 교안 폴더 안의 정답).
_PROMPTS_PATH = Path("data/prompts.yaml")
if not _PROMPTS_PATH.exists():
    _PROMPTS_PATH = Path("../data/prompts.yaml")
if not _PROMPTS_PATH.exists():
    _PROMPTS_PATH = Path("../../data/prompts.yaml")
PROMPTS = yaml.safe_load(_PROMPTS_PATH.read_text(encoding="utf-8"))


def get_prompt(name, version):
    """이름·버전으로 프롬프트 문자열을 돌려준다(로컬 버전 사전에서)."""
    return PROMPTS[name][version]


## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다.

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다
gym = pd.read_csv('data/gym_members.csv')
print('회원 수:', len(gym))
print(gym.head())
gym_summary = summarize_by(gym, '회원권', '월방문횟수')
print(gym_summary)

## 1. 성찰 루프 조립: 임계 점수로 멈추기
**배경**: 제공된 `generate`·`critique`·`revise` 를 묶어 **생성 → (비평 → 수정) 반복** 루프를 만듭니다 (생성·비평·수정의 조합).

**요구사항**:
- 함수 **`reflect(summary, threshold=8, max_iter=3)`** 를 만드세요.
- `generate(summary)` 로 초안을 만든 뒤, 최대 `max_iter` 회 동안 `critique` 로 점수를 매겨 리스트에 모으고, 점수가 `threshold` 이상이면 멈추고, 아니면 `revise` 로 고쳐 반복합니다.
- 반환은 `(최종 리포트, 점수 이력 리스트)` 입니다.
- 그 함수를 `gym_summary` 로 한 번 호출해 결과를 **`final_report`** 와 **`score_history`** 에 담으세요(자가채점이 이 두 이름을 봅니다).
- 루프 안에서는 준비 셀이 만들어 둔 **`generate`·`critique`·`revise` 를 이름 그대로** 부르세요(기본 인자로 받아 두면 자가채점이 부품을 가짜로 바꿔 확인할 때 걸립니다).

**예시**: `reflect(gym_summary)` 의 점수 이력은 `[낮은 점수, ..., 8 이상]` 처럼 오르거나 최대 3개까지입니다.

> 자가채점은 `reflect(gym_summary, threshold=1, max_iter=1)` 을 **직접 한 번 더 호출**해 루프가 실제 `generate`·`critique` 를 부르는지 확인합니다(모델 호출 2회). 그러니 **인자 이름과 기본값을 요구사항대로** 지키세요.

<details><summary>힌트</summary>

```text
접근방법:
- 초안을 만들고, 반복하며 비평 점수를 모으고, 임계 도달 시 break.

세부구현:
1. 요약으로 초안을 한 번 만들고, 점수를 모을 빈 리스트를 준비한다.
2. 최대 반복 횟수만큼 돌면서 초안을 비평하고, 나온 점수를 리스트에 더한다.
3. 점수가 임계값에 닿으면 그 자리에서 멈추고, 아니면 개선점을 넘겨 초안을 다시 쓴 뒤 이어 돈다.
4. (report, score_history) 를 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(score_history, list) and 1 <= len(score_history) <= 3
assert all(isinstance(s, int) and 1 <= s <= 10 for s in score_history)
assert score_history[-1] >= 8 or len(score_history) == 3   # 임계 도달 또는 최대 반복
assert isinstance(final_report, str) and len(final_report.strip()) >= 30, (
    '최종 리포트는 세 문장짜리 글입니다 - 짧은 글자를 손으로 적어 넣으면 여기서 걸립니다')

# 값은 손으로 적어도 위 검사를 통과합니다. 그래서 '진짜' 부품으로 reflect 를 한 번 더 돌려
# 루프가 실제로 굴러가는지 봅니다(임계 1 · 최대 1회 = 생성 1번 + 비평 1번, 모델 호출 두 번).
live_report, live_history = reflect(gym_summary, threshold=1, max_iter=1)
assert len(live_history) == 1 and isinstance(live_history[0], int) and 1 <= live_history[0] <= 10
assert isinstance(live_report, str) and len(live_report.strip()) >= 30, (
    'reflect 가 실제 generate·critique 를 불러 리포트를 만들어야 합니다')

# 이어서 부품 세 개를 잠시 '점수를 미리 정해 둔 가짜'로 바꿔 reflect 를 한 번 더 돌려,
# 루프가 정말 그 순서로 도는지 봅니다(모델 호출 없음).
real_parts = (generate, critique, revise)
probe_calls = []
probe_scores = iter([4, 6, 9])

def generate(summary):
    probe_calls.append('generate')
    return '초안'

def critique(report):
    probe_calls.append('critique')
    return Critique(score=next(probe_scores), issues=['수치 추가'])

def revise(report, issues):
    probe_calls.append('revise')
    return report + '+수정'

try:
    probe_report, probe_history = reflect('요약')
finally:
    generate, critique, revise = real_parts

assert probe_history == [4, 6, 9], '점수가 임계값에 닿을 때까지 비평 점수를 모두 모아야 합니다'
assert probe_calls == ['generate', 'critique', 'revise', 'critique', 'revise', 'critique'], (
    '초안은 한 번만 만들고, 임계 미달일 때만 수정한 뒤 다시 비평해야 합니다')
assert isinstance(probe_report, str) and probe_report.strip()
print('✅ 통과!')

## 2. 성찰 루프 조립: 정해진 횟수만큼
**배경**: 이번엔 임계값 없이 **정확히 정해진 횟수**만큼 비평·수정을 반복합니다(다른 종료 조건).

**요구사항**:
- 함수 **`reflect_fixed(summary, rounds)`** 를 만드세요.
- `generate` 로 초안을 만든 뒤, **정확히 `rounds` 번** `critique` → `revise` 를 반복하며 점수를 모읍니다 (임계값으로 중간에 멈추지 않습니다).
- 반환은 `(최종 리포트, 점수 이력)` 이고 점수 이력의 길이는 **정확히 `rounds`** 입니다.
- 그 함수를 `reflect_fixed(gym_summary, 2)` 로 호출해 결과를 **`rep2`** 와 **`hist2`** 에 담으세요(자가채점이 이 두 이름을 봅니다). 1번과 마찬가지로 부품은 **이름 그대로** 부릅니다.

**예시**: `reflect_fixed(gym_summary, 2)` 의 점수 이력 길이는 **2** 입니다.

> 1번과 같이 자가채점이 `reflect_fixed(gym_summary, 1)` 을 **직접 한 번 더 호출**합니다(모델 호출 3회: 생성·비평·수정).

<details><summary>힌트</summary>

```text
접근방법:
- 1번과 비슷하되 break 없이 rounds 번 모두 critique→revise 를 돈다.

세부구현:
1. 초안을 한 번 만들고, 점수를 모을 빈 리스트를 준비한다.
2. 정해진 횟수만큼 빠짐없이 돌면서 비평하고 점수를 모은 뒤, 개선점을 넘겨 다시 쓴다(중간에 멈추지 않는다).
3. (report, history) 를 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(hist2) == 2
assert all(isinstance(s, int) and 1 <= s <= 10 for s in hist2)
assert isinstance(rep2, str) and len(rep2.strip()) >= 30, (
    '최종 리포트는 세 문장짜리 글입니다 - 짧은 글자를 손으로 적어 넣으면 여기서 걸립니다')

# 1번처럼, 먼저 '진짜' 부품으로 한 회차만 돌려 루프가 실제로 굴러가는지 봅니다
# (rounds=1 = 생성 1 + 비평 1 + 수정 1, 모델 호출 세 번).
live_report2, live_hist2 = reflect_fixed(gym_summary, 1)
assert len(live_hist2) == 1 and isinstance(live_hist2[0], int) and 1 <= live_hist2[0] <= 10
assert isinstance(live_report2, str) and len(live_report2.strip()) >= 30, (
    'reflect_fixed 가 실제 generate·critique·revise 를 불러 리포트를 만들어야 합니다')

# 이어서 부품을 가짜로 바꿔 '임계값과 무관하게 rounds 번을 다 도는지' 확인합니다.
real_parts = (generate, critique, revise)
probe_calls = []

def generate(summary):
    probe_calls.append('generate')
    return '초안'

def critique(report):
    probe_calls.append('critique')
    return Critique(score=10, issues=['수치 추가'])   # 만점이어도 멈추면 안 된다

def revise(report, issues):
    probe_calls.append('revise')
    return report + '+수정'

try:
    probe_report, probe_history = reflect_fixed('요약', 3)
finally:
    generate, critique, revise = real_parts

assert probe_history == [10, 10, 10], '점수가 높아도 rounds 번을 모두 돌아야 합니다'
assert probe_calls.count('critique') == 3 and probe_calls.count('revise') == 3
assert probe_calls[0] == 'generate' and probe_calls.count('generate') == 1
assert isinstance(probe_report, str) and probe_report.strip()
print('✅ 통과!')

## 3. 개선 이력 로그 만들기
**배경**: 반복별 점수를 **로그**로 남기면 개선 과정을 추적할 수 있습니다(관측성의 축소판).

**요구사항**:
- 다른 요약(`summarize_by(gym, '성별', '만족도')`)으로 `reflect_fixed(..., 2)` 를 돌려 그 점수 이력을 **`sat_history`** 에 담고, 그것을 **`[{'회차': 1, '점수': ...}, {'회차': 2, '점수': ...}]`** 형태의 리스트 **`score_log`** 로 만드세요.

**예시**: `score_log` 는 길이 2 이고 각 원소에 `'회차'`·`'점수'` 키가 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- reflect_fixed 의 점수 이력을 enumerate 로 돌며 딕셔너리 리스트로 바꾼다.

세부구현:
1. summarize_by(gym, '성별', '만족도') 로 요약을 만든다.
2. reflect_fixed(요약, 2) 의 점수 이력을 받는다.
3. enumerate(이력, start=1) 로 {'회차': i, '점수': s} 리스트를 만들어 score_log 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(sat_history) == 2 and all(isinstance(s, int) and 1 <= s <= 10 for s in sat_history)
assert len(score_log) == 2
assert all('회차' in row and '점수' in row for row in score_log)
assert [row['회차'] for row in score_log] == [1, 2], '회차는 1 부터 순서대로여야 합니다'
# 로그의 점수가 실제 실행 이력과 같은지 - 손으로 지어낸 점수는 여기서 걸린다
assert [row['점수'] for row in score_log] == sat_history
print('✅ 통과!')

## 4. 분석 도구 + 생성 결합
**배경**: 데이터에서 요약을 뽑아 바로 리포트를 생성합니다(분석과 생성의 조합).

**요구사항**:
- `summarize_by(gym, '회원권', '재등록여부')` 로 만든 요약 문자열을 **`renew_summary`** 에 담고, 그것을 `generate` 로 넘겨 만든 리포트 초안을 **`renew_report`** 에 담으세요.

**예시**: `renew_report` 는 비어 있지 않은 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- summarize_by 결과를 generate 에 그대로 넣는다.

세부구현:
1. summarize_by(gym, '회원권', '재등록여부') 로 요약을 만든다.
2. 그 요약을 renew_summary 에 담는다.
3. generate(renew_summary) 를 renew_report 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 요약은 계산으로 확인할 수 있다 - 여기서 분석 단계를 건너뛴 답이 걸린다
assert renew_summary == summarize_by(gym, '회원권', '재등록여부')
assert isinstance(renew_report, str) and len(renew_report.strip()) >= 30
assert renew_report != renew_summary, '요약을 그대로 두지 말고 generate 로 리포트를 만들어야 합니다'
print('✅ 통과!')

## 5. 관측 콜백 부착하기
**배경**: 호출에 관측 콜백을 붙이면 실행이 Langfuse 대시보드로 자동 전송됩니다. **고치는 곳은 호출 로직이 아니라 `config` 한 자리**라는 점을 확인하세요.

**요구사항**:
- 함수 **`observed_ask(chat_model, question)`** 를 만드세요. 받은 모델로 질문을 부르되 준비 셀의 **`handlers`** 를 `config={'callbacks': handlers}` 로 함께 넘기고, **응답 객체를 그대로** 돌려줍니다.
- 그 함수를 `observed_ask(model, '헬스장 재등록률을 높이는 방법 한 가지')` 로 불러 응답 객체를 **`cb_response`** 에, 그 `.text` 를 **`cb_answer`** 에 담으세요.
- 모델을 **인자로** 받는 이유는 자가채점이 **가짜 모델**을 넣어 `config` 를 정말 넘겼는지 확인하기 때문입니다(실제 호출 없이 확인합니다).
- 응답 객체를 남겨 두는 이유는 **토큰 수**(`usage_metadata['total_tokens']`)를 확인하기 위해서입니다. 자가채점이 그 값이 0 보다 큰지를 봅니다(실제 호출이 있었다는 증거이자, 그 호출이 대시보드로 간 그 호출입니다).

**예시**: `cb_answer` 는 비어 있지 않은 문자열이고, 같은 호출이 Langfuse 의 **Traces** 목록에 한 건 올라옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 받은 모델로 부르되, config 자리에 준비 셀의 handlers 를 실어 보낸다.

세부구현:
1. 함수는 모델과 질문을 받아 그 모델의 invoke 를 부른다.
2. invoke 에 config 를 함께 넘긴다 - 키는 'callbacks', 값은 준비 셀의 handlers 다.
3. 응답 객체를 그대로 돌려주고, 부른 쪽에서 .text 를 꺼내 cb_answer 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 관측 콜백 준비 - 실행만 하세요.
# 핸들러는 .env 의 LANGFUSE_* 를 스스로 읽습니다 - 키를 인자로 넘기지 않습니다.
import os

from langfuse import get_client
from langfuse.langchain import CallbackHandler

# 키를 먼저 확인합니다 - 핸들러를 만든 뒤에 검사하면 이 안내가 묻힙니다.
if not (os.getenv('LANGFUSE_PUBLIC_KEY') and os.getenv('LANGFUSE_SECRET_KEY')):
    raise RuntimeError('Langfuse 키를 찾지 못했습니다. 일차 폴더 .env 의 LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY 를 채우고 커널을 재시작하세요.')

handlers = [CallbackHandler()]

# 키가 '있지만 틀린' 경우 langfuse 는 전송만 조용히 실패합니다 - 그래서 인증을 여기서 확인합니다.
try:
    authenticated = get_client().auth_check()
except Exception:
    authenticated = False
if not authenticated:
    raise RuntimeError('Langfuse 인증에 실패했습니다. 일차 폴더 .env 의 키와 LANGFUSE_BASE_URL 을 확인하고 커널을 재시작하세요.')

print('Langfuse 관측 켜짐 - 이제부터의 호출이 대시보드로 전송됩니다')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(handlers, list) and handlers, '준비 셀의 handlers 를 그대로 쓰세요'
assert isinstance(handlers[0], CallbackHandler), '준비 셀이 만든 콜백 핸들러를 그대로 쓰세요'
assert isinstance(cb_answer, str) and len(cb_answer.strip()) >= 10
# 답 문자열은 손으로 적어도 위 검사를 통과합니다 - 응답에 붙어 온 토큰 수로 실제 호출을 확인합니다.
assert cb_answer == cb_response.text, 'cb_answer 에는 cb_response 의 본문을 담으세요'
assert cb_response.usage_metadata.get('total_tokens', 0) > 0, (
    '실제 호출이라면 응답에 토큰 수가 함께 옵니다 - 문자열을 손으로 적으면 여기서 걸립니다')

# 콜백을 정말 붙였는지는 결과만 봐서는 알 수 없습니다 - 가짜 모델을 넣어 config 를 들여다봅니다
# (실제 모델 호출은 일어나지 않습니다).
from types import SimpleNamespace

seen = {}


def probe_invoke(question, **kwargs):
    seen['config'] = kwargs.get('config')
    return SimpleNamespace(text='가짜 응답', usage_metadata={'total_tokens': 1})


observed_ask(SimpleNamespace(invoke=probe_invoke), '점검용 질문')
assert seen.get('config'), 'observed_ask 안에서 invoke 에 config 를 함께 넘겨야 합니다'
assert seen['config'].get('callbacks') is handlers, (
    "config={'callbacks': handlers} 로 준비 셀의 handlers 를 그대로 넘기세요")
print('✅ 통과! (전송 여부는 대시보드에서 눈으로 확인하세요)')

## 6. 프롬프트 버전 교체 실험
**배경**: 같은 입력에 프롬프트 **v1·v2** 를 각각 적용해 결과를 비교합니다(프롬프트 관리와 생성의 조합).

**요구사항**:
- `get_prompt('insight_writer', 'v1')` 과 `'v2'` 로 꺼낸 프롬프트를 각각 **`system_v1`**, **`system_v2`** 에 담으세요.
- 그 둘을 각각 system 으로, `gym_summary` 를 user 로 넣어 호출한 결과 문자열을 각각 **`out_v1`**, **`out_v2`** 에 담으세요.
- 각 호출 응답의 `usage_metadata['total_tokens']` 도 각각 **`tokens_v1`**, **`tokens_v2`** 에 담으세요 (자가채점이 그 값이 0 보다 큰지를 봅니다. 버전 비교는 **비용 비교**이기도 합니다).

**예시**: 두 결과 모두 비어 있지 않은 문자열이고, 토큰 수는 둘 다 0 보다 큽니다(v2 는 대개 더 구조적이고 더 깁니다).

<details><summary>힌트</summary>

```text
접근방법:
- 버전마다 system 프롬프트를 꺼내 같은 user 로 invoke 하고, 응답에서 본문과 토큰을 함께 꺼낸다.

세부구현:
1. system_v1 = get_prompt('insight_writer', 'v1'), system_v2 = ... 로 두 프롬프트를 꺼낸다.
2. model.invoke([system, user]) 의 반환 응답을 변수에 담아, .text 를 out_v1 에 담는다.
3. 같은 응답의 usage_metadata 에서 total_tokens 를 tokens_v1 에 담는다(없을 때를 대비해 .get 을 쓴다).
4. v2 도 같은 방식으로 out_v2·tokens_v2 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 프롬프트를 레지스트리에서 실제로 꺼냈는지 - 손으로 적었거나 같은 버전을 두 번 썼으면 걸린다
assert system_v1 == get_prompt('insight_writer', 'v1')
assert system_v2 == get_prompt('insight_writer', 'v2')
assert system_v1 != system_v2
assert isinstance(out_v1, str) and len(out_v1.strip()) >= 30
assert isinstance(out_v2, str) and len(out_v2.strip()) >= 30
assert out_v1 not in (system_v1, gym_summary) and out_v2 not in (system_v2, gym_summary)
# 출력은 길이만 보면 아무 문자열이나 통과합니다 - 응답에 붙어 온 토큰 수로 실제 호출을 확인합니다.
assert isinstance(tokens_v1, int) and tokens_v1 > 0, '실제 호출이라면 v1 응답에 토큰 수가 함께 옵니다'
assert isinstance(tokens_v2, int) and tokens_v2 > 0, '실제 호출이라면 v2 응답에 토큰 수가 함께 옵니다'
print('✅ 통과!')

## 7. 폴백 구성: 실패하면 대체 호출로
**배경**: 한 호출이 실패하면 **다른 호출로 폴백**합니다. LiteLLM 의 `fallbacks` 가 하는 일을 직접 구현해 봅니다. (여기서는 폴백 **로직 자체**가 확인 대상이라, 실제 장애를 기다리는 대신 **반드시 실패하는 로컬 함수**로 실패 시점을 우리가 정합니다. 매번 같은 결과가 나와야 채점할 수 있기 때문입니다.)

**요구사항**:
- 함수 **`call_with_fallback(primary, fallback)`** 를 만드세요. `primary()` 를 시도하고, 예외가 나면 `fallback()` 의 결과를 돌려줍니다(둘 다 실패하면 예외가 그대로 나도 됩니다).

**예시**: `primary` 가 예외를 던지면 `fallback()` 의 반환값이 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- try 에서 primary(), except 에서 fallback() 을 반환한다.

세부구현:
1. 두 함수를 인자로 받아, try 블록에서 1차 함수를 실행해 결과를 돌려준다.
2. except 로 실패를 붙잡았을 때 2차 함수를 실행해 그 결과를 돌려준다(둘 다 '실행은 이 안에서' 한다).
```

</details>

In [ ]:
# 잘못된 키 호출을 흉내 내는 함수(항상 실패)와 정상 폴백 함수
def bad_key_call():
    raise RuntimeError('인증 실패(401): 잘못된 키')

def good_key_call():
    return '정상 키로 응답 성공'

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert call_with_fallback(bad_key_call, good_key_call) == '정상 키로 응답 성공'
assert call_with_fallback(good_key_call, bad_key_call) == '정상 키로 응답 성공'

# 위 두 줄은 정해진 문자열을 그냥 돌려주기만 해도 통과합니다 - 누가 언제 불렸는지를 기록해 확인합니다.
probe_calls = []

def probe_fail():
    probe_calls.append('primary')
    raise RuntimeError('인증 실패(401)')

def probe_ok():
    probe_calls.append('fallback')
    return '대체 응답'

assert call_with_fallback(probe_fail, probe_ok) == '대체 응답'
assert probe_calls == ['primary', 'fallback'], '1차를 먼저 부르고, 실패했을 때만 폴백을 불러야 합니다'

probe_calls.clear()

def probe_first():
    probe_calls.append('primary')
    return '1차 응답'

assert call_with_fallback(probe_first, probe_ok) == '1차 응답'
assert probe_calls == ['primary'], '1차가 성공하면 폴백은 부르지 않아야 합니다'
print('✅ 통과!')

## 8. 견고한 호출 래퍼: 재시도 → 폴백 → 실패 보고
**배경**: 재시도와 폴백을 한 함수로 묶고, 그래도 안 되면 **실패를 보고**합니다(예외·재시도·폴백의 통합).

**요구사항**:
- 함수 **`robust_call(call, num_retries=2, fallback=None)`** 를 만드세요.
  - `call()` 을 최대 `num_retries + 1` 번 시도하고, 성공하면 `{'ok': True, 'value': 결과, 'attempts': 시도횟수}` 를 돌려줍니다.
  - 모두 실패하고 `fallback` 이 있으면 `fallback()` 을 시도해 성공 시 `{'ok': True, 'value': ..., 'used_fallback': True}` 를 돌려줍니다.
  - 폴백도 없거나 실패하면 `{'ok': False, 'error': 마지막에러}` 를 돌려줍니다.

**예시**: 항상 실패하는 호출 + 폴백 없음 → `{'ok': False, ...}`. 폴백 있음 → `{'ok': True, 'used_fallback': True, ...}`.

<details><summary>힌트</summary>

```text
접근방법:
- 반복으로 재시도하고, 실패가 이어지면 폴백, 그래도 안 되면 실패 딕셔너리.

세부구현:
1. 시도 횟수만큼 반복하며 매 회차를 try 로 감싼다. 성공하면 그 자리에서 성공 딕셔너리를 반환하고, 실패하면 에러 메시지만 남기고 다음 회차로 넘어간다(요구사항의 키 구성을 그대로 맞출 것).
2. 반복을 다 쓰고도 내려왔다면 폴백 차례다. 폴백이 있을 때만 시도하고, 성공하면 폴백을 썼다는 표시를 함께 담는다.
3. 여기까지 왔으면 실패다. 실패 표시와 마지막 에러를 담아 돌려준다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 반환 딕셔너리만 보면 호출을 한 번도 안 하는 껍데기도 통과합니다 - 시도 횟수를 세어 확인합니다.
probe_calls = []

def probe_fail():
    probe_calls.append('call')
    raise RuntimeError('429 쿼터 초과')

def probe_ok():
    probe_calls.append('fallback')
    return '대체 응답'

fail = robust_call(probe_fail, num_retries=2)
assert fail['ok'] is False and 'error' in fail
assert probe_calls == ['call'] * 3, 'num_retries=2 면 첫 시도 + 재시도 2회 = 총 3번 불러야 합니다'

probe_calls.clear()
recovered = robust_call(probe_fail, num_retries=1, fallback=probe_ok)
assert recovered['ok'] is True and recovered.get('used_fallback') is True
assert recovered['value'] == '대체 응답'
assert probe_calls == ['call', 'call', 'fallback'], '재시도를 다 쓴 뒤에 폴백을 불러야 합니다'

probe_calls.clear()

def probe_good():
    probe_calls.append('call')
    return '정상 응답'

ok = robust_call(probe_good, num_retries=0)
assert ok['ok'] is True and ok['attempts'] == 1 and ok['value'] == '정상 응답'
assert probe_calls == ['call'], '성공하면 그 자리에서 끝내야 합니다'

# 두 번째에 성공하는 호출로 attempts 를 확인합니다 - 1 로 고정해 두면 여기서 걸립니다.
probe_calls.clear()


def probe_flaky():
    probe_calls.append('call')
    if len(probe_calls) < 2:
        raise RuntimeError('일시적 오류')
    return '두 번째에 성공'


retried = robust_call(probe_flaky, num_retries=2)
assert retried['ok'] is True and retried['value'] == '두 번째에 성공'
assert retried['attempts'] == 2, 'attempts 는 실제로 부른 횟수입니다'
print('✅ 통과!')

## 9. 프롬프트를 Langfuse 에 올리고 라벨로 불러오기
**배경**: 6번은 프롬프트를 **코드 옆 딕셔너리**에 뒀습니다. 실제 운영에서는 교안 4절처럼 **서버**에 올려 두고 이름·라벨로 불러옵니다. 그래야 코드를 다시 배포하지 않고도 프롬프트를 바꿀 수 있습니다(관측·프롬프트 버전관리의 조합).

**요구사항**:
- `langfuse = get_client()` 로 클라이언트를 얻으세요(준비 셀에서 이미 `get_client` 를 임포트했습니다).
- `langfuse.create_prompt(name=PROMPT_NAME, prompt=..., labels=['production'])` 로 프롬프트를 올리세요. 본문에는 채울 자리 **`{{summary}}`** 를 반드시 넣습니다.
- `langfuse.get_prompt(PROMPT_NAME, label='production', cache_ttl_seconds=0)` 로 되불러 **`live_prompt`** 에 담고, 그 버전 번호를 **`live_version`** 에 담으세요.
- `live_prompt.compile(summary=gym_summary)` 로 값을 채운 문자열을 **`filled_prompt`** 에 담으세요.

**예시**: `live_version` 은 1 이상의 정수이고, `filled_prompt` 에는 `{{summary}}` 대신 `gym_summary` 의 내용이 들어 있습니다.

> 같은 이름으로 다시 올리면 **덮어쓰기가 아니라 새 버전**입니다. 이 셀을 여러 번 실행하면 대시보드 **Prompts** 탭에 버전이 쌓이는데, 그게 정상입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 4.2(올리기) → 4.3(불러오기) → 4.4(compile) 순서를 그대로 따른다.

세부구현:
1. get_client() 로 클라이언트를 얻는다.
2. create_prompt 에 이름·본문·labels 를 넘긴다. 본문의 채울 자리는 중괄호 두 겹이다.
3. get_prompt 에 label 과 cache_ttl_seconds=0 을 함께 넘겨 방금 올린 판을 받는다.
4. 받은 프롬프트의 version 속성을 꺼내고, compile 에 summary 를 넘겨 채운다.
```

</details>

In [ ]:
# [제공 코드] 이 문제에서 쓸 프롬프트 이름 - 실행만 하세요.
PROMPT_NAME = 'gym-report-writer'

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(live_version, int) and live_version >= 1, '버전 번호는 1 이상의 정수입니다'
assert isinstance(filled_prompt, str) and '{{summary}}' not in filled_prompt, (
    'compile 로 {{summary}} 자리를 채워야 합니다')
assert gym_summary[:20] in filled_prompt, 'filled_prompt 에 gym_summary 내용이 들어가야 합니다'

# 서버에서 한 번 더 불러 대조합니다 - 딕셔너리로 흉내 낸 답안은 여기서 걸립니다.
again = langfuse.get_prompt(PROMPT_NAME, label='production', cache_ttl_seconds=0)
assert again.version == live_version, '서버의 production 버전과 live_version 이 같아야 합니다'
assert 'summary' in again.variables, "프롬프트 본문에 {{summary}} 자리를 넣으세요"
print('✅ 통과! 서버 버전:', again.version)

---
수고했어요! LV2 에서 성찰 루프를 **조립**하고, 프롬프트를 서버에 올려 관리하고, 견고한 호출을 만들었습니다. LV3 에서는 이것을 **하나의 자동 리포트 시스템**으로 통합합니다.